# <span style="color:#1D4ED8">Before class | Reinforcement learning in 15 minutes</span>
**ALU — Software Engineering Specialisation Coach (Machine Learning)**

**Time budget: 15 minutes total, including opening the notebook.** This is the **only required preparation**. You need basic Python familiarity, not previous reinforcement learning experience.

**By the end:** identify state/action/reward, read one Q-table entry, and predict a Q-learning update. If you cannot complete the preparation, attend anyway: the live session begins with a three-minute recap.

## <span style="color:#1D4ED8">0–2 min | Open and run</span>
**Google Colab (recommended):** visit https://colab.research.google.com/ → **Upload** → choose this `.ipynb` → run code cells with **Shift+Enter**. Or open locally in JupyterLab/VS Code with a Python 3 kernel.

Read the Markdown prompts, run the two short code examples, and answer the final three questions. **Do not install packages during the 15-minute preparation**; the code here uses standard Python.

## <span style="color:#1D4ED8">2–4 min · An analogy: training a dog</span>

Imagine rewarding a dog when it sits after a cue. A treat is **positive feedback**, while no treat provides different feedback. Through repeated experiences, the dog may favour actions that lead to rewards. **This is an analogy for reward-guided learning, not a literal Q-learning implementation:** real animals do not necessarily store a numerical Q-table.

**Contrast:** In **supervised learning**, a model learns from examples with known target labels. In **reinforcement learning (RL)**, an **agent** selects actions, observes consequences and receives rewards, aiming to maximize **expected cumulative reward**, not merely the next immediate reward. In **unsupervised learning**, a model discovers structure without target labels supplied for each example.

**Key characteristics of RL:** sequential decisions; delayed consequences; exploration versus exploitation; feedback through rewards; and a policy that can improve through experience.

**Think:** Why might a reward received later affect which action is worth choosing now?

## <span style="color:#1D4ED8">4–9 min · Vocabulary, robot story and Q-table</span>

Our shared story: a robot moves on a **3×3 grid** from **Start (0,0)** to **Goal (2,2)** while avoiding the blocked centre **(1,1)**. Actions are **Up, Down, Left, Right**. A successful move into the goal earns **+10**; every other step, including an invalid move, earns **−1**. An invalid move leaves the robot in its current state.

- **Agent:** the robot making choices.
- **Environment:** the grid and its movement/reward rules.
- **State (s):** the robot's current position, e.g. `(0,0)`.
- **Action (a):** one of the four movement choices.
- **Reward (r):** immediate numerical feedback from the environment.
- **Policy (π):** a rule for selecting actions given states.
- **Episode:** a sequence of actions ending at the goal or a time limit.
- **Q-value Q(s,a):** an estimate of the expected discounted return after taking action `a` in state `s` and following a subsequent policy. **Q-learning aims to learn optimal action values** using experience.
- **Q-table:** a lookup table with **states as rows**, **actions as columns**, and estimated Q-values in cells. A larger Q-value means an action is currently estimated to be better; it does not guarantee that estimate is correct.

**Visual grid:** `S · ·` / `· # ·` / `· · G`.

### Illustrative Q-table (not trained output)

| State | Up | Down | Left | Right |
|---|---:|---:|---:|---:|
| (0,0) Start | 0.0 | 1.0 | 0.0 | **2.0** |
| (0,1) | 1.0 | −2.0 | 0.5 | **6.0** |
| (2,1) Near goal | 0.0 | 4.0 | 1.0 | **7.0** |

**Predict:** From `(0,0)`, which action would a greedy policy choose? What if those numbers are wrong because the robot has hardly explored?

### Five characteristics of reinforcement learning (read alongside the robot example)

**Sequential decisions.** The agent chooses actions repeatedly. Each action changes its state and therefore the choices and outcomes that follow. Robot example: moving Right changes the robot’s position; it must then choose another move. A good decision accounts for where it leads, not only the present moment.

**Delayed consequences (read the Q-table above).** At **(0,0)**, moving **Right** has a current estimate of **Q((0,0), Right) = 2.0**. The robot reaches **(0,1)** and receives **−1** for this nonterminal step. At (0,1), the largest illustrative next-state Q-value is **6.0** (Right). Thus the nonterminal learning target is **−1 + 0.9 × 6 = 4.4**. With learning rate **α = 0.5**, the Start–Right estimate changes to **2 + 0.5 × (4.4 − 2) = 3.2**. The action can be valuable despite its immediate cost because it leads to a state with valuable future options. With **γ = 0**, the target is only **−1** and the same update gives **0.5**. These are *illustrative teaching values*, not results of training; the +10 goal reward motivates why future outcomes can matter.

**Exploration versus exploitation.** Exploration tries actions to learn their consequences; exploitation chooses actions that currently look best. Robot example: it might try Down even when Right has a higher estimated Q-value, to learn whether another route is better. In epsilon-greedy selection, a random action is chosen with probability epsilon (ε); otherwise an action with maximum estimated Q-value is chosen.

**Feedback through rewards.** After an action, the environment returns a numerical reward. This feedback helps the agent revise its action-value estimates; it does not directly provide a labelled “correct action” for every state. Robot example: ordinary moves receive −1 and reaching the goal receives +10. One immediate reward alone does not establish which action is best long term.

**A policy that can improve through experience.** A policy (π) specifies how actions are selected in each state. Tabular Q-learning updates estimated action values from experienced transitions; a greedy policy derived from the Q-table can then change and sometimes improve. Robot example: initially all Q-values may be zero; after training, actions leading toward the goal may have larger estimates. Improvement is not guaranteed in every run.

**Quick check:** Which characteristic explains why a robot might accept a −1 step to reach a +10 goal later? Which explains why it sometimes tries an unfamiliar action?

*Time guidance: this explanation is part of the existing 3–8 minute vocabulary reading, not additional required preparation.*

In [1]:
# A tiny, runnable Q-table reading activity (illustrative values, NOT trained output)
q_demo = {(0,0): [0.0, 1.0, 0.0, 2.0],
          (0,1): [1.0, -2.0, 0.5, 6.0],
          (2,1): [0.0, 4.0, 1.0, 7.0]}
actions = ["Up", "Down", "Left", "Right"]
for state, values in q_demo.items():
    best = actions[values.index(max(values))]
    print(f"State {state}: {dict(zip(actions, values))} -> greedy choice: {best}")


State (0, 0): {'Up': 0.0, 'Down': 2.0, 'Left': 0.0, 'Right': 1.0} -> greedy choice: Down
State (0, 1): {'Up': 1.0, 'Down': -2.0, 'Left': 0.5, 'Right': 3.0} -> greedy choice: Right
State (2, 1): {'Up': 0.0, 'Down': 4.0, 'Left': 1.0, 'Right': 7.0} -> greedy choice: Right


## <span style="color:#0F766E">9–12 min · The mathematics in one sentence</span>

**Intuition:** New estimate = Old estimate + Learning rate × (Target − Old estimate).

For a **nonterminal** transition, tabular Q-learning uses:

$$Q(s,a) \leftarrow Q(s,a) + \alpha\big[r+\gamma\max_{a'}Q(s',a')-Q(s,a)\big].$$

- **α (alpha):** learning rate; how strongly new evidence changes the old estimate.
- **γ (gamma):** discount factor; how much estimated **future** reward matters.
- **r:** reward received now; **s′:** state reached after the action.
- **max Q(s′,a′):** highest estimated next-state action value, not the current state's maximum.
- **Target:** `r + gamma × best_next_Q`. If the next state is **terminal**, target = `r` because there is no further action to value.

You do **not** need to memorize the equation for preparation. In the live session, you will compute one update and see its effect on a visible Q-table.

**Link to the table above:** take `s=(0,0)`, `a=Right`, `s′=(0,1)`, `Q(s,a)=2`, and `max Q(s′,·)=6`. This is a nonterminal move with reward −1. The live notebook uses these exact illustrative values again.

In [2]:
# Preview the Start → Right update from the illustrative Q-table above
old_q, alpha, reward, gamma, best_next_q = 2.0, 0.5, -1.0, 0.9, 6.0
target = reward + gamma * best_next_q
new_q = old_q + alpha * (target - old_q)
print(f"Old={old_q:.1f}, target={target:.1f}, new={new_q:.1f}")


Old=2.0, target=4.4, new=3.2


## <span style="color:#15803D">12–15 min · Prep check (three questions)</span>

1. In the illustrative Q-table, which action has the highest estimated value at **Start**?
2. If **γ = 0**, does the agent consider estimated future rewards or only immediate reward in its Q-learning target?
3. Why might the agent sometimes try an action whose current Q-value is lower than its favourite?

<details><summary>Self-check guidance (open after answering)</summary>

1. **Right** (2.0), based on the *illustrative* row. 2. The nonterminal target becomes the **immediate reward only**. 3. **Exploration** can discover actions whose values are currently underestimated. A greedy choice is based on estimates, not certainty.

</details>

**Bring to class:** one uncertainty, or one prediction about the robot's best route. **No installation is required to understand the prep**; if a code cell fails, read the displayed table and continue.

**Optional Reading, before or after class:** Sutton & Barto, *Reinforcement Learning: An Introduction*, 2nd ed.: http://incompleteideas.net/book/the-book-2nd.html